In [ ]:
# Load Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import argparse
import logging
from sklearn.model_selection import train_test_split, RepeatedKFold, RandomizedSearchCV
from sklearn.feature_selection import RFECV, RFE
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
def parse_arguments():
    """Parse command line arguments"""
    parser = argparse.ArgumentParser(description='Machine Learning Pipeline for Feature Selection and Model Training')
    
    # Input arguments
    parser.add_argument('--input', '-i', 
                    default='../trainingdata/score18h.tsv',
                    help='Input training data file (default: ./data/1.training/3.trainingdata.tsv)')
    # Output arguments
    parser.add_argument('--output-dir', '-o',
                    default='.',
                    help='Output directory for results (default: current directory)')
    
    return parser.parse_args([])

In [ ]:
#interaction_data = pd.read_csv('./data/1.training/3.trainingdata.tsv', delimiter='\t')

In [ ]:
def main():

    # Setup logging
    logging.info("Starting Phage Training Pipeline...")

    # Parse command line arguments
    args = parse_arguments()
    
    # Create output directory if it doesn't exist
    os.makedirs(args.output_dir, exist_ok=True)
    logging.info(f"Output directory: {args.output_dir}")
    
    # Construct full output paths
    feature_plot_path = os.path.join(args.output_dir, 'feature_reduction.png')
    predictions_path = os.path.join(args.output_dir, 'predictions.csv')
    feature_importance_path = os.path.join(args.output_dir, 'feature_importances.csv')
    model_path = os.path.join(args.output_dir, 'best_xgb_model.pkl')
    bacteria_features_path = os.path.join(args.output_dir, 'feature_bacteria.txt')
    phage_features_path = os.path.join(args.output_dir, 'feature_phage.txt')

    
    # Load data
    logging.info(f"Loading data from: {args.input}")
    try:
        interaction_data = pd.read_csv(args.input, delimiter='\t')
        logging.info(f"Data loaded successfully. Shape: {interaction_data.shape}")
    except Exception as e:
        logging.error(f"Failed to load data: {e}")
        return
    
    # Model parameters
    random_state = 3164139949  # for reproducibility
    cores = -2  # -1 uses all cores
    ratio_test = 0.25  # Train/Test ratios

    # RepeatedKFold parameters
    n_splits = 5
    n_repeats = 2

    # REF
    perc_step_1 = 1500
    perc_step_2 = 100
    n_features_final_1 = 3194
    n_features_final_2 = 94

    logging.info("Initializing model components...")
    logging.info(f"Model parameters: random_state={random_state}, cores={cores}, test_ratio={ratio_test}")
    logging.info(f"Cross-validation: {n_splits} splits, {n_repeats} repeats")
    logging.info(f"Feature selection (first part): step={perc_step_1}, final_features={n_features_final_1}")
    logging.info(f"Feature selection (second part): step={perc_step_2}, final_features={n_features_final_2}")

    gradient_boosting = GradientBoostingRegressor(n_estimators=1000, random_state=random_state)

    cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

    # Initialize RFECV with GradientBoostingRegressor
    # This will perform recursive feature elimination with cross-validation
    # The step parameter controls how many features to remove at each iteration
    # The min_features_to_select parameter ensures that at least n_features_final features are selected
    logging.info("Initializing RFECV...")
    rfecv_1 = RFECV(gradient_boosting,
                step=perc_step_1, 
                verbose=100,
                min_features_to_select=n_features_final_1,
                cv=cv,
                n_jobs=cores, 
                scoring='neg_root_mean_squared_error')
    rfecv_2 = RFECV(gradient_boosting,
                step=perc_step_2,
                verbose=100,
                min_features_to_select=n_features_final_2,
                cv=cv,
                n_jobs=cores,
                scoring='neg_root_mean_squared_error')

    # Define the parameter grid for RandomizedSearchCV
    # This grid is based on the hyperparameters of the GradientBoostingRegressor
    # Adjust the values based on your specific needs and computational resources
    param_grid = {
        'n_estimators': [500, 750, 1000, 1250],  # Number of trees in the boosting ensemble
        'max_depth': [3, 5, 10, 20],  # Depth of the individual trees
        'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Learning rate for boosting
        'subsample': [0.8, 0.9, 1.0],  # Fraction of samples used for fitting each tree
        'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split a node
        'min_samples_leaf': [1, 2, 4]  # Minimum number of samples required in each leaf
    }

    # Initialize RandomizedSearchCV with GradientBoostingRegressor
    # This will perform hyperparameter tuning using cross-validation
    # The scoring is set to 'neg_root_mean_squared_error' to minimize RMSE
    # n_iter specifies the number of different combinations to try (remember that the parameters are selected randomly)
    # refit=True means the best model will be refitted on the entire dataset
    logging.info("Initializing RandomizedSearchCV...")
    logging.info(f"Parameter grid size: {len(param_grid)} parameters")
    tuning = RandomizedSearchCV(estimator=GradientBoostingRegressor(random_state=random_state), 
                                param_distributions=param_grid, 
                                cv=cv, 
                                scoring='neg_root_mean_squared_error',
                                verbose=3,
                                n_jobs=cores,
                                n_iter=5,
                                refit=True,
                                random_state=random_state)
    
    
    # Split the data into features (X) and target variable (y)
    X = interaction_data.drop(columns=['Score'])
    y = interaction_data['Score']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ratio_test, random_state=random_state)

    # Perform feature selection using RFECV and filter the training and test feature sets
    logging.info("Starting feature selection with RFECV (first part)...")
    selector_1 = rfecv_1.fit(X_train, y_train)
    X_train_2 = selector_1.transform(X_train)
    X_test_2 = selector_1.transform(X_test)
    logging.info("Starting feature selection with RFECV (second part)...")
    selector_2 = rfecv_2.fit(X_train_2, y_train)
    X_train_final = selector_2.transform(X_train_2)
    X_test_final = selector_2.transform(X_test_2)
    logging.info(f"Selected features: {X_train_final.shape[1]} out of {X_train.shape[1]}")


    # Create feature selection plot
    logging.info(f"Creating feature reduction plot: {feature_plot_path}")
    plt.figure(figsize=(10, 6))
    plt.plot(selector_1.cv_results_['n_features'], selector_1.cv_results_['mean_test_score'], marker='o', linestyle='-', color='b')
    plt.plot(selector_2.cv_results_['n_features'], selector_2.cv_results_['mean_test_score'], marker='o', linestyle='-', color='b')
    plt.title('Model Performance vs Number of Features')
    plt.xlabel('Number of Features')
    plt.ylabel('Cross-validation Score (Neg RMSE)')
    plt.grid(True)
    plt.savefig(feature_plot_path)
    plt.close()
    logging.info("Feature reduction plot saved successfully")

    # Perform hyperparameter tuning using RandomizedSearchCV
    logging.info("Starting hyperparameter tuning...")
    tuning.fit(X_train_final, y_train)
    logging.info(f"Best parameters: {tuning.best_params_}")
    logging.info(f"Best cross-validation score: {tuning.best_score_}")

    # Get the best model from the tuning process
    best_xgb_model = tuning.best_estimator_

    # Attach feature names to the model
    feature_names = selector_1.feature_names_in_[selector_1.support_][selector_2.support_]
    best_xgb_model.feature_names_ = feature_names.tolist()

    # Predict on the test set
    logging.info("Making predictions...")
    y_pred = best_xgb_model.predict(X_test_final)
    y_pred_train = best_xgb_model.predict(X_train_final)

    # Create DataFrames for train and test results and save to CSV
    logging.info(f"Saving predictions to: {predictions_path}")
    train_results = pd.DataFrame({
        'observed': y_train, 
        'predicted': y_pred_train, 
        'dataset':"train"
    })  
    test_results = pd.DataFrame({
        'observed': y_test, 
        'predicted': y_pred , 
        'dataset':"test"
    }) 
    combined_results = pd.concat([train_results, test_results])
    combined_results.to_csv(predictions_path, index=True)
    logging.info("Predictions saved successfully")

    # Calculate RMSE (CURRENTLY NOT OUTPUTTED)
    rmse = np.sqrt(mean_squared_error(np.concatenate([y_pred, y_pred_train]), np.concatenate([y_test, y_train])))
    logging.info(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

    # Feature importance extraction
    feature_importances = best_xgb_model.feature_importances_
    # feature_names = selector_2.feature_names_in_[selector_2.support_]
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importances
    })  
    importance_df.to_csv(feature_importance_path, index=False)
    logging.info("Feature importances saved successfully")

    # Save the best model to a file
    with open(model_path, 'wb') as f:
        pickle.dump(best_xgb_model, f)
    logging.info("Model saved successfully")

    # Filter features for bacteria and phage
    features_bacteria = [feature.replace('_bacteria', '') for feature in feature_names if feature.endswith('_bacteria')]
    features_phage = [feature.replace('_phage', '') for feature in feature_names if feature.endswith('_phage')]
    logging.info(f"Found {len(features_bacteria)} bacteria features and {len(features_phage)} phage features")


    with open(bacteria_features_path, 'w') as file: 
        for feature in features_bacteria:
            file.write(feature + '\n')
    with open(phage_features_path, 'w') as file: 
        for feature in features_phage:
            file.write(feature + '\n')

    logging.info("Pipeline completed successfully!")
    logging.info("=" * 50)
    logging.info("SUMMARY:")
    logging.info(f"  Input file: {args.input}")
    logging.info(f"  Output directory: {args.output_dir}")
    logging.info(f"  Selected features: {X_train_final.shape[1]}")
    logging.info(f"  RMSE: {rmse:.4f}")
    logging.info(f"  Best CV score: {tuning.best_score_:.4f}")
    logging.info("=" * 50)

In [ ]:
if __name__ == "__main__":
    main()